# Aggregations

In [ ]:
# going from clean 2024 data 
import os 
os.getcwd()

'/Users/ellawileman/Documents/last_sem/dsci_capstone/final_repo/US-Census-Voting-and-Registration-DSCI-Capstone-Project/clean_data'

In [ ]:
import pandas as pd
data = pd.read_csv("nov24pub_clean.csv") # replace with your own path to clean data on your machine

In [56]:
data = data.rename(columns={
    "PRTAGE": "age",
    "PESEX": "sex",
    "PEMARITL": "marital_status",
    "PEEDUCA": "education",
    "HEFAMINC": "family_income",
    "PRCHLD": "number_of_children",
    'PTDTRACE': 'race',
    'PEAFEVER': 'veteran',
    'PWSSWGT': 'weight'

})

In [57]:
# subset only a few features for now, can add more in later
data_subset = data[["age","sex", "marital_status","education","family_income","number_of_children","race","veteran","weight"]]
data_subset

,age,sex,marital_status,education,family_income,number_of_children,race,veteran,weight
0,73,2,1,39,10,0,1,2,18354163
1,76,1,1,43,10,0,1,1,15592437
2,85,2,3,39,15,0,1,2,16356249
3,67,1,1,40,16,0,1,2,20773863
4,66,2,1,40,16,0,1,2,16796510
...,...,...,...,...,...,...,...,...,...
62404,69,1,1,40,16,0,1,2,7651462
62405,65,2,1,39,16,0,1,2,6827271
62406,66,1,1,41,12,0,1,2,3872403
62407,73,1,1,39,13,0,1,2,2790829


In [58]:
bins = [18, 25, 35, 45, 55, 65, 100]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]

data_subset["age_group"] = pd.cut(
    data_subset["age"],
    bins=bins,
    labels=labels,
    right=False
)

/var/folders/yz/btvl331d4g54624wdzgx_2hc0000gn/T/ipykernel_48611/2537900120.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_subset["age_group"] = pd.cut(


In [59]:
data_subset

,age,sex,marital_status,education,family_income,number_of_children,race,veteran,weight,age_group
0,73,2,1,39,10,0,1,2,18354163,65+
1,76,1,1,43,10,0,1,1,15592437,65+
2,85,2,3,39,15,0,1,2,16356249,65+
3,67,1,1,40,16,0,1,2,20773863,65+
4,66,2,1,40,16,0,1,2,16796510,65+
...,...,...,...,...,...,...,...,...,...,...
62404,69,1,1,40,16,0,1,2,7651462,65+
62405,65,2,1,39,16,0,1,2,6827271,65+
62406,66,1,1,41,12,0,1,2,3872403,65+
62407,73,1,1,39,13,0,1,2,2790829,65+


In [60]:
# encode the features we don't want to group by in the rows
data_subset = data_subset.drop(columns=["age"])
dummies_df = pd.get_dummies(data_subset, columns=["marital_status", "education","family_income", "number_of_children","veteran"])

# convert the dummies into binary variables for grouping
dummies_df = dummies_df.apply(lambda x: (x > 0).astype(int) if x.name.startswith(('marital_status', 'education', 'family_income', 'number_of_children', 'veteran')) else x)

# multiply the weight column by the dummy variables to get the weighted count for each group
for col in dummies_df.columns:
    if col.startswith(('marital_status', 'education', 'family_income', 'number_of_children', 'veteran')):
        dummies_df[col] = dummies_df[col] * dummies_df["weight"]

# for each group, the proportion of each category is the sum of the weighted counts for that category divided by the total weight for that group
grouped = dummies_df.groupby(["age_group", "sex", "race"]).sum()

/var/folders/yz/btvl331d4g54624wdzgx_2hc0000gn/T/ipykernel_48611/2089487947.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = dummies_df.groupby(["age_group", "sex", "race"]).sum()


In [61]:
grouped.head(50)

weight  marital_status_1  marital_status_2  \
age_group sex race                                                     
18-24     1   1      80372636809        3872165356         452350124   
              2      14489127983          69422956          63508286   
              3       1763439136         208965923          32797217   
              4       5064508524         147015017                 0   
              5        337806600                 0                 0   
              6       1950320541                 0                 0   
              7        991909117                 0                 0   
              8        933797208                 0         142689252   
              9         22065372                 0                 0   
              10       204437291          98601014                 0   
              11       116944431                 0                 0   
              12               0                 0                 0   
              13               0                 0                 0   
              14               0                 0                 0   
              15        42040020                 0                 0   
              16               0                 0                 0   
              17        11636891                 0                 0   
              18        16915965                 0                 0   
              19       165867555                 0                 0   
              20               0                 0                 0   
              21        74585747                 0                 0   
              22               0                 0                 0   
              25               0                 0                 0   
              26        19815342                 0                 0   
          2   1      81344106704        6923348629         371415784   
              2      16050045849         598857379         179611434   
              3       1709370767          88582451                 0   
              4       5622075321         200910032          62502685   
              5        676219668                 0                 0   
              6       1856062071         135387304                 0   
              7       1193848987          93902799                 0   
              8       1526668863                 0                 0   
              9         59646116                 0                 0   
              10        82068009                 0                 0   
              11       226977842                 0                 0   
              12               0                 0                 0   
              13               0                 0                 0   
              14               0                 0                 0   
              15        13484180                 0                 0   
              16        95706802                 0                 0   
              17               0                 0                 0   
              18               0                 0                 0   
              19               0                 0                 0   
              20               0                 0                 0   
              21       101942287           7973263                 0   
              22               0                 0                 0   
              25               0                 0                 0   
              26        17435763                 0                 0   
25-34     1   1     120817176653       45958301433        1226121480   
              2      22046754556        4678115103         123018133   

                    marital_status_3  marital_status_4  marital_status_5  \
age_group sex race                                                         
18-24     1   1             25493792         212244545         433153816   
              2             69083703                 

In [62]:
grouped_nozeros = grouped[grouped["weight"] > 0]
grouped_nozeros

weight  marital_status_1  marital_status_2  \
age_group sex race                                                    
18-24     1   1     80372636809        3872165356         452350124   
              2     14489127983          69422956          63508286   
              3      1763439136         208965923          32797217   
              4      5064508524         147015017                 0   
              5       337806600                 0                 0   
...                         ...               ...               ...   
65+       2   10      244367834         101323456                 0   
              13       24041704                 0                 0   
              15       35975924          21954224                 0   
              16      131158944          38764318                 0   
              21      113188985         104096077                 0   

                    marital_status_3  marital_status_4  marital_status_5  \
age_group sex race                                                         
18-24     1   1             25493792         212244545         433153816   
              2             69083703                 0         189805379   
              3                    0           8072252                 0   
              4                    0                 0          43720166   
              5                    0                 0                 0   
...                              ...               ...               ...   
65+       2   10            34083740          42680279          56131796   
              13                   0          24041704                 0   
              15             7010850           7010850                 0   
              16            69289394          12114653                 0   
              21             9092908                 0                 0   

                    marital_status_6  education_31  education_32  \
age_group sex race                                                 
18-24     1   1          75377229176      39494282     178968033   
              2          14097307659             0             0   
              3           1513603744             0             0   
              4           4873773341             0             0   
              5            337806600             0             0   
...                              ...           ...           ...   
65+       2   10            10148563             0             0   
              13                   0             0             0   
              15                   0             0             0   
              16            10990579             0             0   
              21                   0             0             0   

                    education_33  ...  number_of_children_8  \
age_group sex race                ...                         
18-24     1   1        237524705  ...              13719993   
              2                0  ...                     0   
              3                0  ...                     0   
              4                0  ...                     0   
              5                0  ...                     0   
...                          ...  ...                   ...   
65+       2   10               0  ...                     0   
              13               0  ...                     0   
              15               0  ...                     0   
              16               0  ...                     0   
              21               0  ...                     0   

                    number_of_children_9  number_of_children_10  \
age_group sex race                                                
18-24     1   1                        0                      0   
              2                        0                      0   
              3                        0                      0   
              4                        0                      0   
     

In [63]:
feature_cols = [c for c in grouped_nozeros.columns if c != "weight"]

grouped_nozeros[feature_cols] = grouped_nozeros[feature_cols].div(grouped_nozeros["weight"], axis=0)

/var/folders/yz/btvl331d4g54624wdzgx_2hc0000gn/T/ipykernel_48611/1089809800.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  grouped_nozeros[feature_cols] = grouped_nozeros[feature_cols].div(grouped_nozeros["weight"], axis=0)


In [65]:
grouped_nozeros.head(50)

weight  marital_status_1  marital_status_2  \
age_group sex race                                                     
18-24     1   1      80372636809          0.048178          0.005628   
              2      14489127983          0.004791          0.004383   
              3       1763439136          0.118499          0.018598   
              4       5064508524          0.029028          0.000000   
              5        337806600          0.000000          0.000000   
              6       1950320541          0.000000          0.000000   
              7        991909117          0.000000          0.000000   
              8        933797208          0.000000          0.152805   
              9         22065372          0.000000          0.000000   
              10       204437291          0.482304          0.000000   
              11       116944431          0.000000          0.000000   
              15        42040020          0.000000          0.000000   
              17        11636891          0.000000          0.000000   
              18        16915965          0.000000          0.000000   
              19       165867555          0.000000          0.000000   
              21        74585747          0.000000          0.000000   
              26        19815342          0.000000          0.000000   
          2   1      81344106704          0.085112          0.004566   
              2      16050045849          0.037312          0.011191   
              3       1709370767          0.051822          0.000000   
              4       5622075321          0.035736          0.011117   
              5        676219668          0.000000          0.000000   
              6       1856062071          0.072943          0.000000   
              7       1193848987          0.078656          0.000000   
              8       1526668863          0.000000          0.000000   
              9         59646116          0.000000          0.000000   
              10        82068009          0.000000          0.000000   
              11       226977842          0.000000          0.000000   
              15        13484180          0.000000          0.000000   
              16        95706802          0.000000          0.000000   
              21       101942287          0.078213          0.000000   
              26        17435763          0.000000          0.000000   
25-34     1   1     120817176653          0.380395          0.010149   
              2      22046754556          0.212191          0.005580   
              3       2043611324          0.401418          0.000000   
              4       8348758767          0.255719          0.029122   
              5       1143848068          0.501000          0.000000   
              6       1766394572          0.082663          0.000000   
              7       1159414521          0.446280          0.000000   
              8       1151301808          0.139796          0.000000   
              9         65247311          1.000000          0.000000   
              10        69491276          0.000000          0.000000   
              11       382306396          0.323192          0.000000   
              13        77617227          0.000000          0.000000   
              14       173039894          0.498859          0.501141   
              15        45710024          0.676818          0.000000   
              16       208803234          0.738828          0.000000   
              17        19830150          0.000000          0.000000   
              20        14851643          0.000000          0.000000   
              21        55010538          0.265767          0.000000   

                    marital_status_3  marital_status_4  marital_status_5  \
age_group sex race                                                         
18-24     1   1             0.000317          0.002641          0.005389   
              2             0.004768          0.00000

In [66]:
grouped_nozeros.shape

(195, 57)

In [ ]:
# 200 rows x 20 years ~ 4000 rows 

# we dropped the zero weight rows so this may vary from year to year